# Client segmentation by target rate: SOM and alternative clustering methods

This notebook starts from a **per-`obs_id` dataframe**: one row per observation/client record, with predictor features and a binary target. It compares:

- **SOM**: tune map size and training settings, then cluster the SOM codebook vectors.
- **K-means**: tune the number of clusters and initialisation stability.
- **Gaussian mixture models (GMM)**: tune component count and covariance structure.
- **Agglomerative clustering**: tune cluster count and linkage.

The target is excluded from the feature matrix. Clusters are formed from client characteristics; target rates are calculated afterwards to assess whether the resulting groups differ in outcome incidence. This keeps the clustering unsupervised rather than directly optimising clusters against the target.

The notebook uses internal clustering metrics for parameter selection (primarily silhouette, with Davies–Bouldin and Calinski–Harabasz as supporting diagnostics), then reports target rates, confidence intervals, and feature profiles. Do not interpret a higher target rate as causal.

**Input contract:** a dataframe with one row per `obs_id`, a binary target column, and numeric/categorical predictor columns. If multiple rows exist per `obs_id`, aggregate them before using this notebook.

## 1. Configuration and imports

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from sklearn.decomposition import PCA
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Update these values to match your dataframe ----
DATA_PATH = Path("per_obs_id_features.parquet")  # .parquet or .csv
TARGET = "target_value"
OBS_ID = "obs_id"
CUSTOMER_ID = "cifwithcheckdigit"  # set to None if unavailable

# Columns that must never enter the clustering feature matrix.
EXCLUDE_COLUMNS = {
    OBS_ID, CUSTOMER_ID, TARGET,
    "target_rule", "total_paid", "payment_count",
    "m0_monthenddate", "transactiondate",
}

RANDOM_STATE = 42
MAX_SILHOUETTE_ROWS = 8000  # silhouette is sampled above this size
BOOTSTRAP_REPEATS = 500
MIN_CLUSTER_SIZE = 0.01     # reject solutions with clusters below 1% of rows
MAX_FEATURES_FOR_PLOTS = 20
OUT_DIR = Path("client_cluster_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If you already have the dataframe in memory, replace the loading cell below
# with: df = your_dataframe.copy()


## 2. Load and validate the per-`obs_id` dataframe

In [ ]:
if DATA_PATH.suffix.lower() == ".parquet":
    df = pd.read_parquet(DATA_PATH)
elif DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    raise ValueError("DATA_PATH must point to a .parquet or .csv file.")

if OBS_ID not in df.columns:
    # A saved pandas index is sometimes written as an unnamed CSV column.
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed:")]
    if unnamed:
        df = df.rename(columns={unnamed[0]: OBS_ID})
    else:
        raise ValueError(f"Could not find {OBS_ID!r} in the dataframe.")

if TARGET not in df.columns:
    raise ValueError(f"Target column {TARGET!r} is missing.")

df = df.loc[df[OBS_ID].notna()].copy()
if df[OBS_ID].duplicated().any():
    raise ValueError(
        "Expected exactly one row per obs_id. Aggregate the data to obs_id first."
    )

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.loc[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype("int8")

if CUSTOMER_ID is not None and CUSTOMER_ID in df.columns:
    print("Unique customers:", df[CUSTOMER_ID].nunique())
print("Observations:", f"{len(df):,}")
print("Target prevalence:", f"{df[TARGET].mean():.4f}")
display(df[TARGET].value_counts().rename("count").to_frame())
display(df.head())


## 3. Prepare predictors

Numeric variables are median-imputed and standardised. Categorical variables are mode-imputed and one-hot encoded. Constant and all-missing columns are removed. The target, identifiers, dates, and known outcome-derived fields are excluded.

For very wide data, one-hot encoding can increase dimensionality. The notebook reports the resulting size so this can be reviewed before fitting.

In [ ]:
excluded = set(EXCLUDE_COLUMNS)
if CUSTOMER_ID is not None:
    excluded.add(CUSTOMER_ID)

feature_cols = [c for c in df.columns if c not in excluded]
X_raw = df[feature_cols].copy()
y = df[TARGET].copy()
obs_ids = df[OBS_ID].copy()

# Convert booleans to numeric; remove columns with no usable variation.
for c in X_raw.select_dtypes(include=["bool"]).columns:
    X_raw[c] = X_raw[c].astype("int8")

numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_raw.columns if c not in numeric_cols]

numeric_cols = [
    c for c in numeric_cols
    if X_raw[c].notna().any() and X_raw[c].nunique(dropna=True) > 1
]
categorical_cols = [
    c for c in categorical_cols
    if X_raw[c].notna().any() and X_raw[c].nunique(dropna=True) > 1
]
X_raw = X_raw[numeric_cols + categorical_cols]

transformers = []
if numeric_cols:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    transformers.append(("numeric", numeric_pipe, numeric_cols))
if categorical_cols:
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    transformers.append(("categorical", categorical_pipe, categorical_cols))

if not transformers:
    raise ValueError("No usable predictor columns remain after exclusions.")

preprocessor = ColumnTransformer(transformers, remainder="drop")
X = preprocessor.fit_transform(X_raw)
X = np.asarray(X, dtype=np.float32)

if not np.isfinite(X).all():
    raise ValueError("Preprocessed feature matrix contains non-finite values.")

feature_names = preprocessor.get_feature_names_out()
print("Raw predictors:", X_raw.shape[1])
print("Encoded predictors:", X.shape[1])
print("Matrix shape:", X.shape)


## 4. Shared evaluation functions

Silhouette is the primary tuning criterion. Davies–Bouldin is better when lower; Calinski–Harabasz is better when higher. Scores are computed on a consistent random sample when the dataset is large.

A candidate is rejected if it produces fewer than two clusters, one cluster, or a very small cluster. The minimum-size threshold is configurable above.

In [ ]:
def evaluate_labels(Z, labels, max_rows=MAX_SILHOUETTE_ROWS, seed=RANDOM_STATE):
    labels = np.asarray(labels)
    unique, counts = np.unique(labels, return_counts=True)
    n_clusters = len(unique)
    if n_clusters < 2 or n_clusters >= len(labels):
        return None
    if counts.min() / len(labels) < MIN_CLUSTER_SIZE:
        return None

    if len(Z) > max_rows:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(Z), size=max_rows, replace=False)
        Z_eval = Z[idx]
        labels_eval = labels[idx]
        if len(np.unique(labels_eval)) < 2:
            return None
    else:
        Z_eval, labels_eval = Z, labels

    return {
        "n_clusters": n_clusters,
        "silhouette": silhouette_score(Z_eval, labels_eval),
        "davies_bouldin": davies_bouldin_score(Z_eval, labels_eval),
        "calinski_harabasz": calinski_harabasz_score(Z_eval, labels_eval),
        "min_cluster_share": counts.min() / len(labels),
    }


def make_result_row(method, params, scores):
    return {"method": method, **params, **scores}


def cluster_rate_table(labels, target, ids=None):
    tmp = pd.DataFrame({
        "cluster": np.asarray(labels),
        "target": np.asarray(target),
    }, index=ids)
    out = tmp.groupby("cluster").agg(
        n=("target", "size"),
        positives=("target", "sum"),
        target_rate=("target", "mean"),
    )
    out["share"] = out["n"] / len(tmp)
    out["rate_minus_overall"] = out["target_rate"] - tmp["target"].mean()
    return out.sort_values("target_rate", ascending=False)


def bootstrap_rate_intervals(labels, target, repeats=BOOTSTRAP_REPEATS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    tmp = pd.DataFrame({"cluster": np.asarray(labels), "target": np.asarray(target)})
    rows = []
    for cluster_id, part in tmp.groupby("cluster"):
        vals = part["target"].to_numpy()
        boot = np.array([
            rng.choice(vals, size=len(vals), replace=True).mean()
            for _ in range(repeats)
        ])
        rows.append({
            "cluster": cluster_id,
            "rate": vals.mean(),
            "ci_lower": np.quantile(boot, 0.025),
            "ci_upper": np.quantile(boot, 0.975),
        })
    return pd.DataFrame(rows).set_index("cluster")


## 5. K-means: tune the number of clusters and initialisation

In [ ]:
# Candidate k values adapt to the number of observations.
n = len(X)
k_max = min(15, max(2, int(np.sqrt(n / 2))))
k_values = range(2, k_max + 1)

kmeans_rows = []
kmeans_models = {}
for k in k_values:
    model = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=RANDOM_STATE,
        algorithm="lloyd",
    )
    labels = model.fit_predict(X)
    scores = evaluate_labels(X, labels)
    if scores is not None:
        params = {"k": k, "n_init": 20}
        kmeans_rows.append(make_result_row("kmeans", params, scores))
        kmeans_models[k] = (model, labels)

kmeans_results = pd.DataFrame(kmeans_rows).sort_values(
    ["silhouette", "davies_bouldin"], ascending=[False, True]
)
display(kmeans_results)
if kmeans_results.empty:
    raise RuntimeError("No valid K-means solution. Review feature scaling or MIN_CLUSTER_SIZE.")


## 6. Gaussian mixture models: tune components and covariance structure

GMMs provide soft membership probabilities as well as hard cluster assignments. BIC is used as the model-selection criterion within the GMM family; silhouette and cluster-size checks are also reported for comparability.

In [ ]:
gmm_rows = []
gmm_models = {}
GMM_TUNING_MAX_ROWS = 15000
if n > GMM_TUNING_MAX_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    gmm_idx = np.sort(rng.choice(n, GMM_TUNING_MAX_ROWS, replace=False))
else:
    gmm_idx = np.arange(n)
X_gmm = X[gmm_idx]
component_values = range(2, min(10, max(3, int(np.sqrt(len(X_gmm) / 2)))) + 1)
covariance_types = ["full", "tied", "diag", "spherical"]

for covariance_type in covariance_types:
    for components in component_values:
        try:
            model = GaussianMixture(
                n_components=components,
                covariance_type=covariance_type,
                n_init=3,
                reg_covar=1e-5,
                random_state=RANDOM_STATE,
            )
            labels = model.fit_predict(X_gmm)
            scores = evaluate_labels(X_gmm, labels)
            if scores is None:
                continue
            params = {
                "components": components,
                "covariance_type": covariance_type,
                "bic": model.bic(X_gmm),
                "aic": model.aic(X_gmm),
            }
            gmm_rows.append(make_result_row("gmm", params, scores))
            gmm_models[(components, covariance_type)] = (model, labels)
        except (ValueError, np.linalg.LinAlgError):
            continue

gmm_results = pd.DataFrame(gmm_rows)
if not gmm_results.empty:
    display(gmm_results.sort_values(["bic", "silhouette"], ascending=[True, False]))
else:
    print("No valid GMM candidate. This can happen with high-dimensional or highly collinear data.")


## 7. Agglomerative clustering: tune cluster count and linkage

Ward linkage is evaluated with Euclidean distance. Average and complete linkage are also tested with Euclidean distance. For large datasets, hierarchical clustering can be expensive; the optional cap below allows a reproducible tuning sample. If sampling is used, the final model is refitted on the full dataset only if computationally feasible.

In [ ]:
AGGLOMERATIVE_MAX_ROWS = 12000
if n > AGGLOMERATIVE_MAX_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    agg_idx = np.sort(rng.choice(n, AGGLOMERATIVE_MAX_ROWS, replace=False))
else:
    agg_idx = np.arange(n)

X_agg = X[agg_idx]
agg_rows = []
agg_models = {}
agg_k_max = min(15, max(2, int(np.sqrt(len(X_agg) / 2))))

for linkage in ["ward", "average", "complete"]:
    for k in range(2, agg_k_max + 1):
        try:
            model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            labels = model.fit_predict(X_agg)
            scores = evaluate_labels(X_agg, labels)
            if scores is None:
                continue
            params = {"k": k, "linkage": linkage, "tuning_n": len(X_agg)}
            agg_rows.append(make_result_row("agglomerative", params, scores))
            agg_models[(k, linkage)] = (model, labels)
        except (ValueError, MemoryError):
            continue

agg_results = pd.DataFrame(agg_rows)
if not agg_results.empty:
    display(agg_results.sort_values(
        ["silhouette", "davies_bouldin"], ascending=[False, True]
    ))
else:
    print("No valid agglomerative candidate.")


## 8. SOM: tune map size and training settings

The SOM is trained on the same preprocessed feature matrix. Map dimensions are generated from dataset size rather than fixed in advance. A small candidate set of map sizes, neighbourhood radii, learning rates, and training lengths is evaluated.

For each SOM candidate:
1. Quantisation error measures how closely observations match their best-matching units (BMUs).
2. Topographic error measures how often the first and second BMUs are not adjacent.
3. The codebook vectors are clustered with K-means; the resulting observation labels are evaluated using silhouette.

Install `MiniSom` if it is not already available.

In [ ]:
%pip -q install MiniSom

In [ ]:
from minisom import MiniSom

# Map-size candidates scale with n. Keep the grid compact enough to avoid
# creating many nearly empty neurons.
base_side = max(3, int(round(np.sqrt(np.sqrt(n)))))
map_sides = sorted(set([
    max(3, base_side - 2),
    base_side,
    min(20, base_side + 2),
]))
som_map_shapes = [(side, side) for side in map_sides[:2]]

# Parameter candidates are deliberately varied; the best combination is selected
# using quantisation/topographic error and downstream codebook-cluster silhouette.
som_sigmas = [max(0.8, side / 4) for side in map_sides]
som_learning_rates = [0.2, 0.5]
som_epochs = [20, 50]
som_cluster_counts = sorted(set([2, 3, 4, 5, 6, 8, 10]))
som_cluster_counts = [k for k in som_cluster_counts if k < n]

# Use a deterministic subset for SOM hyperparameter search if n is very large.
SOM_TUNING_MAX_ROWS = 12000
if n > SOM_TUNING_MAX_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    som_idx = np.sort(rng.choice(n, SOM_TUNING_MAX_ROWS, replace=False))
else:
    som_idx = np.arange(n)
X_som_tune = X[som_idx]

som_rows = []
som_candidates = {}
for rows, cols in som_map_shapes:
    for sigma in sorted(set([max(0.8, min(rows, cols) / 4), max(1.0, min(rows, cols) / 2)])):
        for learning_rate in som_learning_rates:
            for epochs in som_epochs:
                som = MiniSom(
                    x=rows, y=cols, input_len=X.shape[1],
                    sigma=sigma, learning_rate=learning_rate,
                    neighborhood_function="gaussian",
                    random_seed=RANDOM_STATE,
                )
                som.random_weights_init(X_som_tune)
                som.train_random(X_som_tune, num_iteration=epochs * len(X_som_tune))

                bmu = np.array([som.winner(row) for row in X_som_tune])
                bmu_flat = bmu[:, 0] * cols + bmu[:, 1]
                qe = som.quantization_error(X_som_tune)
                te = som.topographic_error(X_som_tune)

                codebook = som.get_weights().reshape(rows * cols, X.shape[1])
                # Exclude unused neurons when assigning codebook clusters.
                occupied = np.unique(bmu_flat)
                occupied_vectors = codebook[occupied]

                for k in som_cluster_counts:
                    if k >= len(occupied):
                        continue
                    km_codebook = KMeans(
                        n_clusters=k, n_init=10, random_state=RANDOM_STATE
                    )
                    neuron_labels = km_codebook.fit_predict(occupied_vectors)
                    neuron_to_cluster = dict(zip(occupied, neuron_labels))
                    obs_labels = np.array([neuron_to_cluster[u] for u in bmu_flat])
                    scores = evaluate_labels(X_som_tune, obs_labels)
                    if scores is None:
                        continue

                    params = {
                        "rows": rows, "cols": cols, "sigma": sigma,
                        "learning_rate": learning_rate, "epochs": epochs,
                        "cluster_k": k, "quantization_error": qe,
                        "topographic_error": te,
                    }
                    som_rows.append(make_result_row("som", params, scores))
                    som_candidates[(rows, cols, sigma, learning_rate, epochs, k)] = {
                        "som": som, "km_codebook": km_codebook,
                        "occupied": occupied, "neuron_labels": neuron_labels,
                        "tuning_labels": obs_labels,
                    }

som_results = pd.DataFrame(som_rows)
if som_results.empty:
    print("No valid SOM candidate. Consider reducing MIN_CLUSTER_SIZE or expanding the candidate ranges.")
else:
    display(som_results.sort_values(
        ["silhouette", "quantization_error", "topographic_error"],
        ascending=[False, True, True]
    ).head(20))


## 9. Compare candidate methods and select representative solutions

The top candidate from each family is retained for comparison. Silhouette is the primary ranking metric; the other internal metrics and SOM errors are retained to make trade-offs visible. A single metric should not be treated as proof that a clustering is meaningful.

In [ ]:
candidate_tables = []
for table in [kmeans_results, gmm_results, agg_results, som_results]:
    if table is not None and not table.empty:
        candidate_tables.append(table)

all_results = pd.concat(candidate_tables, ignore_index=True, sort=False)
display(all_results.sort_values(
    ["silhouette", "davies_bouldin"], ascending=[False, True]
).head(30))

best_by_method = {}
if not kmeans_results.empty:
    r = kmeans_results.sort_values("silhouette", ascending=False).iloc[0]
    best_by_method["kmeans"] = kmeans_models[int(r["k"])][1]

if not gmm_results.empty:
    r = gmm_results.sort_values(["bic", "silhouette"], ascending=[True, False]).iloc[0]
    key = (int(r["components"]), r["covariance_type"])
    best_by_method["gmm"] = gmm_models[key][0].predict(X)

if not agg_results.empty:
    r = agg_results.sort_values("silhouette", ascending=False).iloc[0]
    key = (int(r["k"]), r["linkage"])
    # If tuned on a sample, only sample labels are available here.
    best_by_method["agglomerative_sample"] = (agg_idx, agg_models[key][1])

if not som_results.empty:
    r = som_results.sort_values(
        ["silhouette", "quantization_error", "topographic_error"],
        ascending=[False, True, True]
    ).iloc[0]
    key = (
        int(r["rows"]), int(r["cols"]), float(r["sigma"]),
        float(r["learning_rate"]), int(r["epochs"]), int(r["cluster_k"])
    )
    chosen = som_candidates[key]
    chosen_som = chosen["som"]
    chosen_rows, chosen_cols = key[0], key[1]
    occupied = chosen["occupied"]
    neuron_labels = chosen["neuron_labels"]
    neuron_to_cluster = dict(zip(occupied, neuron_labels))
    full_bmu = np.array([chosen_som.winner(row) for row in X])
    full_bmu_flat = full_bmu[:, 0] * chosen_cols + full_bmu[:, 1]
    # A neuron not occupied during tuning is assigned to its nearest occupied
    # codebook vector, so every observation receives a cluster.
    occupied_vectors = chosen_som.get_weights().reshape(
        chosen_rows * chosen_cols, X.shape[1]
    )[occupied]
    all_weights = chosen_som.get_weights().reshape(
        chosen_rows * chosen_cols, X.shape[1]
    )
    for neuron_id in np.unique(full_bmu_flat):
        if neuron_id not in neuron_to_cluster:
            nearest = np.argmin(
                np.linalg.norm(occupied_vectors - all_weights[neuron_id], axis=1)
            )
            neuron_to_cluster[neuron_id] = neuron_labels[nearest]
    best_by_method["som"] = np.array(
        [neuron_to_cluster[u] for u in full_bmu_flat]
    )

print("Selected:", list(best_by_method))


## 10. Compare target rates for selected clusterings

Target rates are calculated only after clustering. The table includes cluster size and the difference from the overall target prevalence. Bootstrap intervals provide a rough measure of uncertainty; they do not account for model-selection uncertainty.

In [ ]:
rate_tables = {}
for method, labels in best_by_method.items():
    if method == "agglomerative_sample":
        idx, sample_labels = labels
        rates = cluster_rate_table(sample_labels, y.iloc[idx].to_numpy(), obs_ids.iloc[idx].to_numpy())
    else:
        rates = cluster_rate_table(labels, y.to_numpy(), obs_ids.to_numpy())
    rate_tables[method] = rates
    print(f"\n{method}")
    display(rates)

# Plot cluster target rates for each selected method.
for method, rates in rate_tables.items():
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ordered = rates.sort_values("target_rate")
    ax.bar(ordered.index.astype(str), ordered["target_rate"])
    ax.axhline(y.mean(), linestyle="--", linewidth=1, label=f"Overall = {y.mean():.3f}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Target rate")
    ax.set_title(f"{method}: target rate by cluster")
    ax.legend()
    plt.tight_layout()
    plt.show()


## 11. Feature profiles and distinguishing characteristics

For each full-data solution, compare cluster means/medians with the overall population. Standardised differences are descriptive. For categorical variables, use encoded feature proportions; for readability, display the strongest features only.

In [ ]:
def feature_profile(Z_frame, labels):
    frame = pd.DataFrame(Z_frame, columns=feature_names)
    frame["cluster"] = np.asarray(labels)
    return frame.groupby("cluster").mean()

profile_methods = {
    name: labels for name, labels in best_by_method.items()
    if name != "agglomerative_sample" and len(labels) == len(df)
}
for method, labels in profile_methods.items():
    prof = feature_profile(X, labels)
    overall = pd.Series(X.mean(axis=0), index=feature_names)
    sd = pd.Series(X.std(axis=0), index=feature_names).replace(0, np.nan)
    differences = (prof - overall) / sd
    print(f"\n{method}: strongest cluster-feature differences")
    for cluster_id in differences.index:
        top = differences.loc[cluster_id].abs().sort_values(ascending=False).head(MAX_FEATURES_FOR_PLOTS).index
        display(differences.loc[[cluster_id], top].T.rename(columns={cluster_id: f"cluster_{cluster_id}"}))


## 12. Optional target-rate significance check

A chi-square test assesses whether target incidence is independent of cluster assignment. This is a descriptive association test, not a causal test. With very large samples, small differences can become statistically significant, so inspect effect sizes and confidence intervals too.

In [ ]:
from scipy.stats import chi2_contingency

association_rows = []
for method, labels in profile_methods.items():
    contingency = pd.crosstab(pd.Series(labels, name="cluster"), y.reset_index(drop=True))
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    n_total = contingency.to_numpy().sum()
    # Cramer's V effect size
    denom = n_total * min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / denom) if denom > 0 else np.nan
    association_rows.append({
        "method": method, "chi2": chi2, "degrees_freedom": dof,
        "p_value": p_value, "cramers_v": cramers_v,
    })

association_results = pd.DataFrame(association_rows)
display(association_results)


## 13. Export assignments, profiles, and tuning results

The exported assignment file maps each `obs_id` to its cluster. It is suitable for joining back to the client-level dataframe. Agglomerative labels are exported separately if tuning used a sample.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_results.to_csv(OUT_DIR / "clustering_tuning_results.csv", index=False)
association_results.to_csv(OUT_DIR / "target_cluster_association.csv", index=False)

for method, labels in profile_methods.items():
    assignments = pd.DataFrame({
        OBS_ID: obs_ids.to_numpy(),
        "cluster": np.asarray(labels),
        TARGET: y.to_numpy(),
    })
    assignments.to_csv(OUT_DIR / f"{method}_assignments.csv", index=False)
    cluster_rate_table(labels, y.to_numpy(), obs_ids.to_numpy()).to_csv(
        OUT_DIR / f"{method}_target_rates.csv"
    )

if "agglomerative_sample" in best_by_method:
    idx, labels = best_by_method["agglomerative_sample"]
    pd.DataFrame({
        OBS_ID: obs_ids.iloc[idx].to_numpy(),
        "cluster": labels,
        TARGET: y.iloc[idx].to_numpy(),
    }).to_csv(OUT_DIR / "agglomerative_sample_assignments.csv", index=False)

print("Saved outputs to:", OUT_DIR.resolve())


## 14. Interpretation checklist

When reviewing the output:

1. Prefer solutions with reasonable silhouette and cluster sizes, but do not choose solely on silhouette.
2. Check whether cluster target rates are meaningfully separated and whether their confidence intervals overlap.
3. Check whether the same broad patterns appear across methods; agreement is evidence of robustness.
4. Inspect feature profiles to understand what distinguishes high- and low-rate groups.
5. Re-run with a different random seed and, where feasible, a different customer sample to assess stability.
6. If multiple `obs_id` rows belong to the same customer, use customer-level resampling or a customer-grouped validation design when quantifying uncertainty.
7. Remember that target rates are observation-weighted here. If the same customer contributes multiple `obs_id` records, also calculate customer-weighted rates or restrict to one observation per customer.

**Important limitation:** unsupervised clusters are not guaranteed to have distinct target rates. If the rates are nearly identical across all reasonable clusterings, the feature space may not contain strong segmentation structure related to the target. A separate supervised segmentation approach would then be a different modelling objective.